# MetroRide with MLflow: track runs, link them to data, register a model

The Spark ML tutorial (`ml/tutorial_spark_ml.ipynb`) kept its results in a Python list that disappears
when the kernel stops. Here every training run is recorded by **MLflow**:

| Step | What you add | MLflow API |
|---|---|---|
| 1 | Point MLflow at a tracking store and pick an experiment | `set_tracking_uri`, `set_experiment` |
| 2 | Read the training data at a fixed **DuckLake snapshot** | (pyducklake) `current_snapshot()` |
| 3 | Wrap `fit_and_score()` in a run | `start_run`, `log_params`, `set_tag`, `log_metrics`, `spark.log_model` |
| 4 | Compare runs, spot the leak | `search_runs`, the MLflow UI |
| 5 | Keep a different test set in its own experiment | `set_experiment` |
| 6 | Register the best model and load it by alias | `register_model`, `set_registered_model_alias`, `models:/name@alias` |
| 7 | Get the exact training rows back from a run | `get_run`, `scan().with_snapshot()` |
| 8 | (Optional) Autologging | `mlflow.pyspark.ml.autolog()` |

**Before you start**

* Run from the `mlops/` folder: `pip install -r requirements.txt` (needs Java 17+ for Spark).
* The notebook builds its own lakehouse in `mlops/lakehouse/` the first time (a few seconds).
* The DuckLake catalog is one DuckDB file: **only one process can open it at a time**. Shut down
  this kernel before running `pipeline.py` or `tutorial_prefect.ipynb`.

## 0. Setup

In [ ]:
import mlflow
from mlflow import MlflowClient
from pyspark.ml import Pipeline
from pyspark.sql import functions as F

import lakehouse as lh          # bronze -> silver -> gold steps (compact phase 1-3 code)
import ml                       # Spark helpers: to_spark, load_training_data, build_stages, ...
from ml import LABEL, CATEGORICAL, REGISTERED_MODEL

spark = ml.get_spark()
print("Spark", spark.version, "| MLflow", mlflow.__version__)

# Build the lakehouse once (all six months), without an orchestrator
trip_context = lh.get_catalog().load_table(("silver", "trip_context"))
if trip_context.scan().count() == 0:
    lh.build_all_months()
lh.row_counts()

## 1. Point MLflow at a tracking store

`setup_mlflow()` uses a SQLite file, `mlops/mlflow.db`, so no server is needed. Model files go to
`mlops/mlartifacts/`. To browse the runs, open a terminal in `mlops/` and run

```
mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000
```

then open http://127.0.0.1:5000. With a shared tracking server instead, set the environment variable
`MLFLOW_TRACKING_URI=http://<server>:5000` before starting Jupyter.

In [ ]:
EXPERIMENT = "metroride-arrival-wait"
print("tracking URI:", ml.setup_mlflow(EXPERIMENT))
print("experiment  :", mlflow.get_experiment_by_name(EXPERIMENT).experiment_id, EXPERIMENT)

## 2. Read the training data at a fixed snapshot

`silver.trip_context` is a DuckLake table, so every commit creates a new snapshot. We **pin** the snapshot
we train on and will store its id on every run. Months later, the same id gives back exactly the same rows.

In [ ]:
SNAPSHOT_ID = ml.current_snapshot_id()
data = ml.load_training_data(SNAPSHOT_ID)
print("snapshot", SNAPSHOT_ID, "|", data.count(), "labelled trips")

train = data.filter(F.col("month") <= 8)       # June-August
test = data.filter(F.col("month") == 9)        # September
print("train:", train.count(), "| test:", test.count())
data.orderBy("trip_id").show(3)

## 3. Wrap `fit_and_score()` in an MLflow run

The Spark ML code is the same as in the tutorial. The added lines only **record** what happened:

* `start_run` opens a run and closes it at the end of the `with` block, even after an error;
* `log_params`: the inputs (feature lists, split);
* `set_tag`: free labels, here the DuckLake snapshot;
* `log_metrics`: the scores;
* `mlflow.spark.log_model`: the whole fitted `PipelineModel`, as a file artifact of the run.

In [ ]:
SPLIT = "train Jun-Aug, test Sep"


def fit_and_score(name, categorical, numeric, train_df=train, test_df=test, split=SPLIT):
    # Fit a pipeline on train_df, score it on test_df, and record everything in one MLflow run.
    with mlflow.start_run(run_name=name) as run:
        mlflow.log_params({"categorical": categorical, "numeric": numeric,
                           "algorithm": "LinearRegression", "split": split})
        mlflow.set_tag("ducklake.snapshot_id", SNAPSHOT_ID)

        model = Pipeline(stages=ml.build_stages(categorical, numeric)).fit(train_df)
        metrics = ml.regression_metrics(model.transform(test_df))

        mlflow.log_metrics(metrics)
        info = mlflow.spark.log_model(model, artifact_path="model")
    print(f"{name:32s} mae {metrics['mae']:.2f}  rmse {metrics['rmse']:.2f}  r2 {metrics['r2']:.2f}"
          f"   run {run.info.run_id[:8]}")
    return model, info

The baseline has no model, but it is still a run: we want its score in the same table.

In [ ]:
with mlflow.start_run(run_name="baseline: training mean"):
    mlflow.log_params({"algorithm": "training mean", "split": SPLIT})
    mlflow.set_tag("ducklake.snapshot_id", SNAPSHOT_ID)
    train_mean = train.agg(F.avg(LABEL)).first()[0]
    baseline = ml.regression_metrics(test.withColumn("prediction", F.lit(train_mean)))
    mlflow.log_metrics(baseline)
print(f"baseline mae {baseline['mae']:.2f}")

In [ ]:
m1, info1 = fit_and_score("zone + slot + weekday", CATEGORICAL, [])
m2, info2 = fit_and_score("+ accept delay", CATEGORICAL, ["accept_delay_minutes"])
leaky, _ = fit_and_score("LEAKY: + passenger pickup wait", CATEGORICAL,
                         ["accept_delay_minutes", "passenger_pickup_wait_minutes"])

## 4. Compare runs

`search_runs` returns a pandas DataFrame with one row per run: params, metrics and tags are columns.
The same view is in the UI: select the runs and press **Compare**.

A perfect score (MAE 0.00, R² 1.0) is a red flag, not a win: the LEAKY run uses the passenger pickup time,
which is only known after the driver arrives.

In [ ]:
runs = mlflow.search_runs(experiment_names=[EXPERIMENT], order_by=["metrics.mae ASC"])
runs[["tags.mlflow.runName", "metrics.mae", "metrics.rmse", "metrics.r2",
      "params.numeric", "tags.ducklake.snapshot_id"]].round(3)

Filter in code, for example to hide suspicious runs:

In [ ]:
mlflow.search_runs(experiment_names=[EXPERIMENT],
                   filter_string="metrics.r2 < 0.99 AND params.algorithm = 'LinearRegression'",
                   order_by=["metrics.mae ASC"])[["tags.mlflow.runName", "metrics.mae"]].round(3)

## 5. A different test set gets its own experiment

Only September has weather. The rain model trains on 1-20 September and tests on 21-30 September, so its MAE
is **not comparable** with the runs above. We put it in a second experiment.

In [ ]:
ml.setup_mlflow("metroride-arrival-wait-september")
sept = data.filter(F.col("month") == 9)
sept_train = sept.filter(F.dayofmonth("service_date") <= 20)
sept_test = sept.filter(F.dayofmonth("service_date") > 20)
SEPT_SPLIT = "train Sep 1-20, test Sep 21-30"

fit_and_score("Sept: accept delay", [], ["accept_delay_minutes"], sept_train, sept_test, SEPT_SPLIT)
rain_model, _ = fit_and_score("Sept: accept delay + rain", [], ["accept_delay_minutes", "is_rainy"],
                              sept_train, sept_test, SEPT_SPLIT)

_ = ml.setup_mlflow(EXPERIMENT)      # back to the main experiment

## 6. Register the best model and point an alias at it

The registry gives a model a **name** and numbered **versions**; each version links back to its run.
An **alias** such as `champion` is a movable pointer. A scoring job loads `models:/<name>@champion`, so
promoting or rolling back a model is one call and the scoring code never changes.

In [ ]:
client = MlflowClient()
mv = mlflow.register_model(info2.model_uri, REGISTERED_MODEL)
client.set_registered_model_alias(REGISTERED_MODEL, "champion", mv.version)
print(f"{REGISTERED_MODEL} version {mv.version} is now @champion (run {mv.run_id[:8]})")

A scoring job needs only the name and the alias:

In [ ]:
champion = mlflow.spark.load_model(f"models:/{REGISTERED_MODEL}@champion")
print("MAE of the loaded champion on September:", round(ml.regression_metrics(champion.transform(test))["mae"], 2))
champion.transform(test).select("trip_id", LABEL, F.round("prediction", 2).alias("prediction")).show(5)

## 7. Get the exact training rows back from a run

The champion's run carries the DuckLake snapshot id as a tag. Time travel to that snapshot returns
the same rows, even if new months have been loaded since.

In [ ]:
champion_run = client.get_run(client.get_model_version_by_alias(REGISTERED_MODEL, "champion").run_id)
snap = int(champion_run.data.tags["ducklake.snapshot_id"])
same_rows = ml.load_training_data(snap)
print("snapshot", snap, "->", same_rows.count(), "labelled trips (same as training:", data.count(), ")")

# The snapshot history of the catalog: every commit is a snapshot
lh.sql("SELECT snapshot_id, snapshot_time, commit_message FROM metroride.snapshots() ORDER BY snapshot_id DESC LIMIT 5")

## 8. (Optional) Autologging

`mlflow.pyspark.ml.autolog()` logs the params of every stage, training metrics and the model whenever
`fit()` runs, with no explicit log calls. Convenient, but you no longer choose what is recorded,
and it does not know your test set or your DuckLake snapshot.

In [ ]:
mlflow.pyspark.ml.autolog(log_models=False)
with mlflow.start_run(run_name="autolog: + accept delay"):
    mlflow.set_tag("ducklake.snapshot_id", SNAPSHOT_ID)
    Pipeline(stages=ml.build_stages(CATEGORICAL, ["accept_delay_minutes"])).fit(train)
mlflow.pyspark.ml.autolog(disable=True)

last = mlflow.search_runs(experiment_names=[EXPERIMENT], max_results=1)
print(len([c for c in last.columns if c.startswith("params.")]), "params logged automatically, e.g.:")
print(sorted(c for c in last.columns if c.startswith("params.") and last[c].notna().all())[:6])

## Exercises

1. Log the random forest from the Spark ML tutorial (Exercise 2) as a run and compare it in the UI.
2. Log the rain threshold (1.0 mm) as a param of the September runs. Rebuild `is_rainy` with 0.5 and 2.0 mm
   and compare the three runs.
3. Register the rain model under its own name, `metroride-arrival-wait-rain`. Why not as a new version of
   `metroride-arrival-wait`?
4. Move the `champion` alias back to an older version, then load `@champion` again. What changed in the
   scoring code?

## Clean up

Release the lakehouse so that another process (for example `pipeline.py`) can open it.

In [ ]:
lh.close_catalog()
spark.stop()